In [ ]:
!pip install pandas

## 1. 학기별 개설교과목정보 데이터 합치기

In [1]:
import os
import glob
import pandas as pd
import re

files = glob.glob("data/*.xls")

all_data = []

for file in files:
    # HTML 테이블 읽기
    df = pd.read_html(file)[0]
    
    # 첫 번째 행을 컬럼명으로 지정
    df.columns = df.iloc[0]   # 0번 행을 컬럼명으로
    df = df.drop(index=0)     # 0번 행 삭제
    
    # 학기 정보 컬럼 추가
    semester = os.path.basename(file).replace("-sis.xls", "")
    df["개설학기정보"] = semester
    
    all_data.append(df)

# 모든 파일 병합
df_all = pd.concat(all_data, ignore_index=True)

print(df_all.head())


0       학년도   학기  소속          학과     과목번호  분반  \
0  2024 학년도  2학기  대학    게페르트국제학부  AAS1001  01   
1  2024 학년도  2학기  대학  아트&테크놀로지학과  AAT2002  01   
2  2024 학년도  2학기  대학  아트&테크놀로지학과  AAT2003  01   
3  2024 학년도  2학기  대학  아트&테크놀로지학과  AAT2004  01   
4  2024 학년도  2학기  대학  아트&테크놀로지학과  AAT2005  01   

0                                  과목명   학점                수업시간/강의실   시간  ...  \
0                              아시아학 개론  3.0  화,목 12:00~13:15 [J107]  3.0  ...   
1              Humanities & creativity  3.0    월 13:30~16:15 [X514]  3.0  ...   
2                Intro to Digital Arts  3.0    월 13:30~16:15 [X513]  3.0  ...   
3          Intro to Creative Computing  3.0  수,금 09:00~10:15 [X513]  3.0  ...   
4  Creative Capstone Project I(캡스톤디자인)  3.0    금 16:30~19:00 [X427]  3.0  ...   

0 시험일자 수강대상     권장학년 수강신청 참조사항 과목 설명          비고  NaN  개설학기정보 HUSS과목 CI과목  
0  NaN  NaN      전학년     1-4학년   NaN         NaN  NaN  2024-2    NaN  NaN  
1  NaN  NaN      전학년     1-4학년   NaN  구) ANT2003  NaN  2024-2    N

In [2]:
df_all.head(3)

,학년도,학기,소속,학과,과목번호,분반,과목명,학점,수업시간/강의실,시간,...,시험일자,수강대상,권장학년,수강신청 참조사항,과목 설명,비고,NaN,개설학기정보,HUSS과목,CI과목
0,2024 학년도,2학기,대학,게페르트국제학부,AAS1001,01,아시아학 개론,3.0,"화,목 12:00~13:15 [J107]",3.0,...,NaN,NaN,전학년,1-4학년,NaN,NaN,NaN,2024-2,NaN,NaN
1,2024 학년도,2학기,대학,아트&테크놀로지학과,AAT2002,01,Humanities & creativity,3.0,월 13:30~16:15 [X514],3.0,...,NaN,NaN,전학년,1-4학년,NaN,구) ANT2003,NaN,2024-2,NaN,NaN
2,2024 학년도,2학기,대학,아트&테크놀로지학과,AAT2003,01,Intro to Digital Arts,3.0,월 13:30~16:15 [X513],3.0,...,NaN,NaN,"2,3,4학년",1-4학년,NaN,구) ANT2004,NaN,2024-2,NaN,NaN


In [3]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9684 entries, 0 to 9683
Data columns (total 30 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   학년도        9684 non-null   object
 1   학기         9684 non-null   object
 2   소속         9684 non-null   object
 3   학과         9684 non-null   object
 4   과목번호       9684 non-null   object
 5   분반         9684 non-null   object
 6   과목명        9684 non-null   object
 7   학점         9502 non-null   object
 8   수업시간/강의실   9374 non-null   object
 9   시간         9318 non-null   object
 10  교수진        9308 non-null   object
 11  수강생수       9684 non-null   object
 12  영어강의       1922 non-null   object
 13  중국어강의      29 non-null     object
 14  승인과목       1028 non-null   object
 15  CU과목       228 non-null    object
 16  홀짝구분       131 non-null    object
 17  국제학생       417 non-null    object
 18  Honors과목   215 non-null    object
 19  공학인증       61 non-null     object
 20  시험일자       0 non-null      obj

In [4]:
df_all["개설학기정보"].unique()

array(['2024-2', '2025-1', '2023-1', '2025-summer', '2025-2',
       '2025-winter', '2024-1', '2023-winter', '2023-summer', '2023-2',
       '2024-winter', '2024-summer', '2026-1'], dtype=object)

## 2. 필요한 열 선택

In [6]:
def make_major_df(df_all, codes=None, prefix=None):
    if codes is not None:
        df_major = df_all[df_all["과목번호"].isin(codes)]
    elif prefix is not None:
        df_major = df_all[df_all["과목번호"].str.startswith(prefix)]
    else:
        raise ValueError("codes 또는 prefix 중 하나를 지정해야 합니다.")

    result = []
    for code in df_major["과목번호"].unique():
        subset = df_major[df_major["과목번호"] == code]
        sems = ", ".join(sorted(subset["개설학기정보"].unique()))
        names = ", ".join(sorted(subset["과목명"].unique()))
        profs = ", ".join(sorted(subset["교수진"].dropna().unique()))
        result.append({
            "과목번호": code,
            "과목명": names,
            "교수": profs,
            "개설학기": sems
        })
    return pd.DataFrame(result)


## 3. 변수명 만들기 자동화

In [7]:
def make_major_dfs(df_all, major_name, codes=None, prefix=None):
    import builtins

    # 정리된 전공 데이터프레임 생성
    df_major = make_major_df(df_all, codes=codes, prefix=prefix)
    builtins.__dict__[f"df_{major_name}"] = df_major

    # 계절별 데이터 생성
    builtins.__dict__[f"df_{major_name}_spring"] = df_major[df_major["개설학기"].str.contains("-1", na=False)]
    builtins.__dict__[f"df_{major_name}_fall"] = df_major[df_major["개설학기"].str.contains("-2", na=False)]
    builtins.__dict__[f"df_{major_name}_summer"] = df_major[df_major["개설학기"].str.contains("-summer", case=False, na=False)]
    builtins.__dict__[f"df_{major_name}_winter"] = df_major[df_major["개설학기"].str.contains("-winter", case=False, na=False)]


### 4-1. 빅데이터사이언스 연계전공

In [8]:
biksa_codes = [
    "STS2011", "MAT2110", "MAT3020", "MGT2002", "ECO2004", 
    "BDS4010", "CSW4010", "CSE4187",
    "AIC4012", "BDS3010", "BDS3020", "CSE4130", "CSW2010", "CSW2020",
    "CSW2030", "CSE3080", "CSW2050", "CSW3010", "CSE3081", "CSW3030", "CSE4110",
    "CSW3060", "CSW3080",
    "CSW4020", "ECO2009", "ECO3022", "ECO3023", "ECO4003", "ECO4004",
    "ECO4032", "EEE4178", "JAS4014", "MAS1004", "MAS2009",
    "MAS2010", "MAT3110", "MAT4331", "MGT4202", "MGT4208", "MGT4226",
    "MGT4515", "MGT4517", "MGT6613", "MGTG613"
]

make_major_dfs(df_all, major_name="biksa", codes=biksa_codes)

In [9]:
df_biksa_spring

,과목번호,과목명,교수,개설학기
2,CSE3080,자료구조,"소정민, 양지훈, 장부루, 장형수, 정성원, 정영민, 최준석","2023-1, 2023-2, 2024-1, 2024-2, 2024-summer, 2..."
3,CSE3081,알고리즘설계와분석,"김세준, 서양진, 소정민, 임인성, 장형수","2023-1, 2023-2, 2024-2, 2025-2, 2025-summer"
4,CSE4187,"산학프로젝트(캡스톤디자인), 캡스톤디자인II(캡스톤디자인)","구명완, 장두성, 정영민","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
9,ECO2004,경제통계학,"김인경, 박경옥, 백예인, 사공용, 이성원, 이영훈, 주하연","2023-1, 2023-2, 2023-winter, 2024-1, 2024-2, 2..."
10,ECO2009,계량경제학I,"김재호, 남준우, 백예인, 이성원, 이영훈, 이한식","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
14,MAS1004,Data&AI,"김태훈, 남현우, 사영준, 정다샘, 정해동","2023-1, 2023-2, 2024-1, 2024-2, 2024-winter, 2..."
15,MAT2110,선형대수학,"강유태, 김대욱, 김준태, 박준용, 안소영, 옥지훈, 이영란, 조상현, 조성희, 조...","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."
16,MAT3020,통계학입문,"임경수, 임경필","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."
18,MGT2002,경영통계학,"김명석, 김범수, 이군희, 이윤동, 정예숙, 조성빈","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."
19,MGT4202,통계자료분석,이군희,"2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"


In [10]:
df_biksa_summer

,과목번호,과목명,교수,개설학기
2,CSE3080,자료구조,"소정민, 양지훈, 장부루, 장형수, 정성원, 정영민, 최준석","2023-1, 2023-2, 2024-1, 2024-2, 2024-summer, 2..."
3,CSE3081,알고리즘설계와분석,"김세준, 서양진, 소정민, 임인성, 장형수","2023-1, 2023-2, 2024-2, 2025-2, 2025-summer"
15,MAT2110,선형대수학,"강유태, 김대욱, 김준태, 박준용, 안소영, 옥지훈, 이영란, 조상현, 조성희, 조...","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."
16,MAT3020,통계학입문,"임경수, 임경필","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."
18,MGT2002,경영통계학,"김명석, 김범수, 이군희, 이윤동, 정예숙, 조성빈","2023-1, 2023-2, 2023-summer, 2023-winter, 2024..."


In [11]:
df_biksa_fall

,과목번호,과목명,교수,개설학기
0,BDS3020,블록체인및응용,문현아,"2023-2, 2024-2, 2025-2"
1,BDS4010,빅데이터종합설계(캡스톤디자인),"최수진, 하정욱","2023-2, 2024-2, 2025-2"
2,CSE3080,자료구조,"소정민, 양지훈, 장부루, 장형수, 정성원, 정영민, 최준석","2023-1, 2023-2, 2024-1, 2024-2, 2024-summer, 2..."
3,CSE3081,알고리즘설계와분석,"김세준, 서양진, 소정민, 임인성, 장형수","2023-1, 2023-2, 2024-2, 2025-2, 2025-summer"
4,CSE4187,"산학프로젝트(캡스톤디자인), 캡스톤디자인II(캡스톤디자인)","구명완, 장두성, 정영민","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
5,CSW2020,기초Java언어,강호석,"2023-2, 2024-2, 2025-2"
6,CSW2030,자료구조입문,임종석,"2023-2, 2024-2, 2025-2"
7,CSW2050,코퍼스언어학,홍정하,"2023-2, 2024-2, 2025-2"
8,CSW3030,데이터베이스입문,박석,"2023-2, 2024-2, 2025-2"
9,ECO2004,경제통계학,"김인경, 박경옥, 백예인, 사공용, 이성원, 이영훈, 주하연","2023-1, 2023-2, 2023-winter, 2024-1, 2024-2, 2..."


### 4-2. 경영

In [12]:
make_major_dfs(df_all, major_name="mgt", prefix="MGT")

In [68]:
df_mgt_fall

,과목번호,과목명,교수,개설학기
0,MGT2002,경영통계학,"김명석, 김범수, 이군희, 이윤동, 정예숙, 조성빈","2023-1, 2023-2, 2023-summer, 2023-winter, 2024-1, 2024-2, 2024-summer, 2024-winter, 2025-1, 2025-2, 2025-summer"
1,MGT2003,회계학원론,"강평경, 김민, 김아람, 박두리, 박재형, 변상혁, 송민섭, 양준선, 이다혜, 천준범","2023-1, 2023-2, 2023-summer, 2023-winter, 2024-1, 2024-2, 2024-summer, 2024-winter, 2025-1, 2025-2, 2025-summer"
2,MGT3001,조직행동이론,"양동훈, 이인석, 이해경, 장영균, 조봉순, 최장호","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
3,MGT3002,경영정보시스템,"김용진, 김태연, 오혜림, 이상근, 한재형","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
4,MGT3003,"생산관리론, 운영관리","강세원, 김길선, 김민균, 김수효, 하병천","2023-1, 2023-2, 2024-1, 2024-2, 2024-summer, 2025-1, 2025-2"
5,MGT3004,재무관리,"김도성, 박영석, 안성필, 오동석, 이상호, 이영주, 이정진, 홍광헌","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2, 2025-summer"
6,MGT3005,관리회계,"김민, 박예연, 박재형, 변상혁, 양준선, 황국재","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
7,MGT3006,마케팅원론,"김주영, 김채영, 김태민, 박경도, 박세훈, 성연진, 이민경, 전성률, 정재학, 조혜원","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
8,MGT3007,국제경영론,"강리브가, 김도의, 김창수, 박영수, 이강표, 정라미, 정선욱","2023-1, 2023-2, 2023-summer, 2024-1, 2024-2, 2025-1, 2025-2"
9,MGT3008,전공진로설계Ⅰ,"김진화, 양준선, 장영균, 정선욱, 하병천","2024-1, 2024-2"


### 4-3. 국문

In [14]:
make_major_dfs(df_all, major_name="kor", prefix="KOR")

In [15]:
df_kor_fall

,과목번호,과목명,교수,개설학기
0,KOR2001,국어학입문,"이정훈, 이지영","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
1,KOR2200,국문학개설,"백진우, 이정원, 황윤정","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
2,KOR2201,문학이란무엇인가,"김경수, 박숙자, 박슬기, 우찬제","2023-1, 2023-2, 2024-1, 2024-2, 2025-1, 2025-2"
3,KOR3007,국어담화화용론,명정희,"2023-2, 2024-2, 2025-2"
4,KOR3015,국어사,이지영,"2023-1, 2024-2"
5,KOR3201,한문I,백진우,"2024-2, 2025-2"
6,KOR3204,국문학과전통문화,최지혜,"2023-2, 2024-2, 2025-2"
7,KOR3305,고전시가론,최지혜,"2023-2, 2024-2, 2025-2"
8,KOR3402,현대소설텍스트읽기I,김경수,"2023-1, 2024-2"
9,KOR3413,영상문학론,박미란,"2023-2, 2024-2"


### 4-4. 공공인재 연계전공

In [56]:
pub_codes = [
    "PUB2005", "POL3130", "POL2002", "SOC2001", "SOC2003", "SOC3010",
    "ECO2001", "ECO2002", "ECO3009", "ECO3011", "ECO3017", "MGT2002", 
    "MGT2003", "PHI2005", "PSY2001", "PSY3009", "PUB3030", "PUB3016",
    "PUB3029", "PUB3023", "PUB3024", "PUB3025", "PUB3031", "PUB3032",
    "PUB3026", "PUB3021", "PUB3020", "PUB3022", "PUB3028", "PUB3027",
    "PUB4009", "KOR4500", "EDU2001", "PHI4010", "ECO2007", "MGT3004",
    "MGT4301", "MGT3005", "MGT4404"
]

make_major_dfs(df_all, major_name="pub", codes=pub_codes)


### 4-5. 교직

In [57]:
edu_codes = [
    "EDU2001", "EDU2002", "EDU2003", "EDU2004", "EDU3001", "EDU3047",
    "EDU3002", "EDU3033", "EDU3045", "EDU3046", "EDU3037", "EDU3004", "EDU3035",
    "EDU2005", "EDU2007", "EDU3038", "EDU3039", "EDU3048", "EDU3049",
    "SHU4019", "SHU4022", "SHU4031", "EDU3036"
]

make_major_dfs(df_all, major_name="edu", codes=edu_codes)


## 5. 이번학기 시간표 만드는 함수

In [64]:
def make_timetable(df_all, df_major_fall, taken_courses):
    if '개설학기' not in df_all.columns and '개설학기정보' in df_all.columns:
        df_all = df_all.rename(columns={'개설학기정보': '개설학기'})

    if '개설학기' not in df_major_fall.columns and '개설학기정보' in df_major_fall.columns:
        df_major_fall = df_major_fall.rename(columns={'개설학기정보': '개설학기'})

    courses_2025_2 = df_major_fall[df_major_fall["개설학기"].str.contains("2025-2", na=False)]

    merged = df_all[
        (df_all["과목번호"].isin(courses_2025_2["과목번호"])) &
        (df_all["개설학기"] == "2025-2")
    ][["과목번호", "과목명", "수업시간/강의실", "교수진"]]

    # 긴 시간대 쪼개기 매핑
    split_times = {
        "09:00~11:45": ["09:00~10:15", "10:30~11:45"],
        "10:30~13:15": ["10:30~11:45", "12:00~13:15"],
        "13:30~16:15": ["13:30~14:45", "15:00~16:15"]
    }

    expanded_rows = []

    # 이미 수강한 과목 제외
    filtered_df = merged[~merged["과목명"].isin(taken_courses)]

    for _, row in filtered_df.iterrows():
        # NaN 방지 처리
        time_room_str = str(row["수업시간/강의실"]) if pd.notna(row["수업시간/강의실"]) else ""

        # 요일 여러 개 추출
        days = re.findall(r"[월화수목금토일]", time_room_str)
        if not days:
            continue

        # 시간 추출
        time_match = re.search(r"\d{2}:\d{2}~\d{2}:\d{2}", time_room_str)
        if not time_match:
            continue

        time = time_match.group()

        # 긴 시간대 쪼개기
        if time in split_times:
            times_to_add = split_times[time]
        else:
            times_to_add = [time]

        # 요일·시간 모두 확장
        for t in times_to_add:
            for day in days:
                expanded_rows.append({
                    "시간": t,
                    "요일": day,
                    "과목(교수)": f"{row['과목명']}({row['교수진']})"
                })

    # DataFrame 생성
    expanded_df = pd.DataFrame(expanded_rows)

    # 요일·시간별 과목 합치기
    timetable = (
        expanded_df.groupby(["시간", "요일"])["과목(교수)"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
    )

    # 요일 순서 맞추기
    요일순서 = ["월", "화", "수", "목", "금"]
    timetable_pivot = timetable.pivot(index="시간", columns="요일", values="과목(교수)").fillna("")
    timetable_pivot = timetable_pivot.reindex(columns=요일순서)

    return timetable_pivot


### 5-1. 사용

In [32]:
hs_taken_courses = [
    "선형대수학", "경영통계학", "경제통계학", "통계학입문", 
    "기초빅데이터프로그래밍", "고급응용C프로그래밍", "인공지능(딥러닝)개론", 
    "웹데이터수집과 텍스트분석(캡스톤디자인)", "Data&AI",
    "회계학원론", "조직행동이론", "마케팅원론",
    "국제경영론", "경영전략",
    "의사결정론"
]

In [36]:
# 내가 포함하고 싶은 경영 과목명 리스트
selected_mgt_courses = [
    "운영관리",
    "재무관리",
    "경영정보시스템",
    "관리회계",
    "기업윤리",
    "경영과학",
    "경영 데이터사이언스"
]

# 경영 과목 필터링
df_mgt_selected = df_mgt_fall[df_mgt_fall["과목명"].isin(selected_mgt_courses)]


In [37]:
# 두 전공 과목 목록 합치기
df_combined_fall = pd.concat([df_biksa_fall, df_mgt_selected], ignore_index=True)

# 합친 전공 과목으로 시간표 생성
timetable_hs = make_timetable(df_all, df_combined_fall, hs_taken_courses)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

timetable_hs

요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,,금융시장의빅데이터분석(캡스톤디자인)(정재식),,금융시장의빅데이터분석(캡스톤디자인)(정재식)
10:30~11:45,"경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 빅데이터종합설계(캡스톤디자인)(하정욱), 자료구조(정성원), 자료구조입문(임종석), 재무관리(김도성)","계량경제학I(남준우), 데이터베이스입문(박석), 알고리즘설계와분석(임인성), 인공지능 자료분석(이윤동)","경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 빅데이터종합설계(캡스톤디자인)(하정욱), 자료구조(정성원), 자료구조입문(임종석), 재무관리(김도성)","계량경제학I(남준우), 데이터베이스입문(박석), 알고리즘설계와분석(임인성), 인공지능 자료분석(이윤동)",블록체인및응용(문현아)
12:00~13:15,"계량경제학I(이성원), 기초Java언어(강호석), 알고리즘설계와분석(장형수)","경영정보시스템(오혜림), 관리회계(변상혁), 산업수학(캡스톤디자인)(김종락), 통계자료분석(이군희)","계량경제학I(이성원), 기초Java언어(강호석), 알고리즘설계와분석(장형수)","경영정보시스템(오혜림), 관리회계(변상혁), 산업수학(캡스톤디자인)(김종락), 통계자료분석(이군희)",블록체인및응용(문현아)
13:30~14:45,"산학프로젝트(캡스톤디자인)(장두성), 산학프로젝트(캡스톤디자인)(정영민)","데이터마이닝(임종섭), 알고리즘설계와분석(김세준)","경영과학(김범수), 기초Java언어(강호석), 재무관리(홍광헌)","데이터마이닝(임종섭), 알고리즘설계와분석(김세준)","경영과학(김범수), 기초Java언어(강호석), 재무관리(홍광헌)"
15:00~16:15,"산학프로젝트(캡스톤디자인)(장두성), 산학프로젝트(캡스톤디자인)(정영민)","계량경제학I(이영훈), 재무관리(이영주), 코퍼스언어학(홍정하)",관리회계(김민),"계량경제학I(이영훈), 재무관리(이영주), 코퍼스언어학(홍정하)",관리회계(김민)
16:30~17:45,관리회계(양준선),경영 데이터사이언스(조성빈),관리회계(양준선),경영 데이터사이언스(조성빈),


In [67]:
import re

# 1. 매핑 만들 때 과목명 전처리
def clean_course_name(name):
    # 괄호 안 내용 제거
    return re.sub(r"\(.*?\)", "", str(name)).strip()

df_all['과목명_clean'] = df_all['과목명'].apply(clean_course_name)
name_to_code = dict(zip(df_all['과목명_clean'], df_all['과목번호']))

# 2. 색칠 함수
def colorize_cell(cell):
    if pd.isna(cell) or cell.strip() == "":
        return cell

    courses = str(cell).split(', ')
    colored_courses = []

    for course_name in courses:
        # 동일하게 전처리
        course_name_clean = clean_course_name(course_name)

        code = name_to_code.get(course_name_clean, None)
        in_biksa = code in biksa_codes_set if code else False
        in_mgt = code in mgt_codes_set if code else False

        if in_biksa and in_mgt:
            colored_courses.append(f'<span style="color:orange">{course_name}</span>')
        elif in_mgt:
            colored_courses.append(f'<span style="color:red">{course_name}</span>')
        elif in_biksa:
            colored_courses.append(f'<span style="color:blue">{course_name}</span>')
        else:
            colored_courses.append(course_name)

    return "<br>".join(colored_courses)

# 3. 적용
timetable_colored = timetable_hs.map(colorize_cell)
from IPython.display import HTML
HTML(timetable_colored.to_html(escape=False))


요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,,금융시장의빅데이터분석(캡스톤디자인)(정재식),,금융시장의빅데이터분석(캡스톤디자인)(정재식)
10:30~11:45,경영과학(민재형)경영정보시스템(김용진)기업윤리(장영균)빅데이터종합설계(캡스톤디자인)(하정욱)자료구조(정성원)자료구조입문(임종석)재무관리(김도성),계량경제학I(남준우)데이터베이스입문(박석)알고리즘설계와분석(임인성)인공지능 자료분석(이윤동),경영과학(민재형)경영정보시스템(김용진)기업윤리(장영균)빅데이터종합설계(캡스톤디자인)(하정욱)자료구조(정성원)자료구조입문(임종석)재무관리(김도성),계량경제학I(남준우)데이터베이스입문(박석)알고리즘설계와분석(임인성)인공지능 자료분석(이윤동),블록체인및응용(문현아)
12:00~13:15,계량경제학I(이성원)기초Java언어(강호석)알고리즘설계와분석(장형수),경영정보시스템(오혜림)관리회계(변상혁)산업수학(캡스톤디자인)(김종락)통계자료분석(이군희),계량경제학I(이성원)기초Java언어(강호석)알고리즘설계와분석(장형수),경영정보시스템(오혜림)관리회계(변상혁)산업수학(캡스톤디자인)(김종락)통계자료분석(이군희),블록체인및응용(문현아)
13:30~14:45,산학프로젝트(캡스톤디자인)(장두성)산학프로젝트(캡스톤디자인)(정영민),데이터마이닝(임종섭)알고리즘설계와분석(김세준),경영과학(김범수)기초Java언어(강호석)재무관리(홍광헌),데이터마이닝(임종섭)알고리즘설계와분석(김세준),경영과학(김범수)기초Java언어(강호석)재무관리(홍광헌)
15:00~16:15,산학프로젝트(캡스톤디자인)(장두성)산학프로젝트(캡스톤디자인)(정영민),계량경제학I(이영훈)재무관리(이영주)코퍼스언어학(홍정하),관리회계(김민),계량경제학I(이영훈)재무관리(이영주)코퍼스언어학(홍정하),관리회계(김민)
16:30~17:45,관리회계(양준선),경영 데이터사이언스(조성빈),관리회계(양준선),경영 데이터사이언스(조성빈),


In [27]:
timetable_kor = make_timetable(df_all, df_kor_fall, hs_taken_courses)

timetable_kor

요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,국어교과교재연구및지도법(강효경),,국어교과교재연구및지도법(강효경),
10:30~11:45,"국어형태론(전지영), 문예창작론(장서란)","국어학입문(이지영), 한문I(백진우)","국어형태론(전지영), 문예창작론(장서란)","국어학입문(이지영), 한문I(백진우)",문학과문화콘텐츠(김예람)
12:00~13:15,"고전문학자료연구(이정원), 국어의문장구조(이정훈)","국어교과논리및논술(강효경), 현대소설텍스트읽기II(김경수)","고전문학자료연구(이정원), 국어의문장구조(이정훈)","국어교과논리및논술(강효경), 현대소설텍스트읽기II(김경수)",문학과문화콘텐츠(김예람)
13:30~14:45,,"국어방언론(김한별), 한문과한문학(백진우)",소설과영화(김예람),"국어방언론(김한별), 한문과한문학(백진우)","국문학과전통문화(최지혜), 소설과영화(김예람)"
15:00~16:15,,"국어학연습(김한별), 이야기와이야기창작(김경수)",국어담화화용론(명정희),"국어학연습(김한별), 이야기와이야기창작(김경수)","국문학과전통문화(최지혜), 국어담화화용론(명정희)"
16:30~17:45,문학이란무엇인가(박슬기),"고전시가론(최지혜), 국문학개설(황윤정)",문학이란무엇인가(박슬기),"고전시가론(최지혜), 국문학개설(황윤정)",


In [29]:
ssong_taken_courses = [
    "문학이란무엇인가", "국어학입문", "국문학개설",
    "국어음운론", "국어사", "국어의미론",
    "현대시텍스트읽기I",
    "설화 문학의 이해",
    "문학과 융합적 상상력"
]

In [30]:
timetable_kor_ssong = make_timetable(df_all, df_kor_fall, ssong_taken_courses)

timetable_kor_ssong

요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,국어교과교재연구및지도법(강효경),,국어교과교재연구및지도법(강효경),
10:30~11:45,"국어형태론(전지영), 문예창작론(장서란)",한문I(백진우),"국어형태론(전지영), 문예창작론(장서란)",한문I(백진우),문학과문화콘텐츠(김예람)
12:00~13:15,"고전문학자료연구(이정원), 국어의문장구조(이정훈)","국어교과논리및논술(강효경), 현대소설텍스트읽기II(김경수)","고전문학자료연구(이정원), 국어의문장구조(이정훈)","국어교과논리및논술(강효경), 현대소설텍스트읽기II(김경수)",문학과문화콘텐츠(김예람)
13:30~14:45,,"국어방언론(김한별), 한문과한문학(백진우)",소설과영화(김예람),"국어방언론(김한별), 한문과한문학(백진우)","국문학과전통문화(최지혜), 소설과영화(김예람)"
15:00~16:15,,"국어학연습(김한별), 이야기와이야기창작(김경수)",국어담화화용론(명정희),"국어학연습(김한별), 이야기와이야기창작(김경수)","국문학과전통문화(최지혜), 국어담화화용론(명정희)"
16:30~17:45,,고전시가론(최지혜),,고전시가론(최지혜),


In [31]:
timetable_mgt_ssong = make_timetable(df_all, df_mgt_fall, ssong_taken_courses)
timetable_mgt_ssong

요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,기술경영과기술사업화(백서현),비즈니스프로세스관리(이준겸),기술경영과기술사업화(백서현),비즈니스프로세스관리(이준겸)
10:30~11:45,"경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 마케팅원론(정재학), 모바일통신경영(이상근), 인공지능과경영모델(김진화), 인공지능과마케팅(김주영), 재무관리(김도성), 조직행동이론(조봉순), 중급회계1(송민섭), 회계학원론(양준선)","경영통계학(이군희), 공급체인관리와e-business(김민균), 국제경영론(김창수), 기업과국제환경(김유진), 리더쉽이론(이인석), 원가회계(박예연), 인공지능 자료분석(이윤동), 투자론(박영석)","경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 마케팅원론(정재학), 모바일통신경영(이상근), 인공지능과경영모델(김진화), 인공지능과마케팅(김주영), 재무관리(김도성), 조직행동이론(조봉순), 중급회계1(송민섭), 회계학원론(양준선)","경영통계학(이군희), 공급체인관리와e-business(김민균), 국제경영론(김창수), 기업과국제환경(김유진), 리더쉽이론(이인석), 원가회계(박예연), 인공지능 자료분석(이윤동), 투자론(박영석)","경영전략(김윤진), 기업과국제환경(김장순)"
12:00~13:15,"국제경영론(정선욱), 노사관계론(양동훈), 디지털변혁과비즈니스모델혁신(김용진), 마케팅원론(박경도), 인적자원관리(조봉순), 조직행동이론(장영균), 중급회계2(송민섭), 파생상품론(이정진)","경영전략(박종훈), 경영정보시스템(오혜림), 관리회계(변상혁), 국제경영론(강리브가), 소비자행동론(성연진), 운영관리(김길선), 중급회계1(강평경), 통계자료분석(이군희), 회계학원론(박두리)","국제경영론(정선욱), 노사관계론(양동훈), 디지털변혁과비즈니스모델혁신(김용진), 마케팅원론(박경도), 인적자원관리(조봉순), 조직행동이론(장영균), 중급회계2(송민섭), 파생상품론(이정진)","경영전략(박종훈), 경영정보시스템(오혜림), 관리회계(변상혁), 국제경영론(강리브가), 소비자행동론(성연진), 운영관리(김길선), 중급회계1(강평경), 통계자료분석(이군희), 회계학원론(박두리)","경영전략(김윤진), 기업과국제환경(김장순)"
13:30~14:45,"국제경영론(정라미), 국제마케팅론(김도의), 무역경영관리(손명옥), 세무회계(천준범), 웹데이터수집과 텍스트분석(캡스톤디자인)(김명석)","경영통계학(이윤동), 금융기관론(이상호), 로지스틱스와생산전략(하병천), 운영관리(강세원), 원가회계(박예연), 의사결정론(조성빈), 이해관계자자본주의와ESG(박영석)","경영과학(김범수), 서비스마케팅(이민경), 재무관리(홍광헌), 파생상품론(원재환), 프로세스 수익경영(이준겸), 회계학원론(김민)","경영통계학(이윤동), 금융기관론(이상호), 로지스틱스와생산전략(하병천), 운영관리(강세원), 원가회계(박예연), 의사결정론(조성빈), 이해관계자자본주의와ESG(박영석)","경영과학(김범수), 서비스마케팅(이민경), 시뮬레이션(정철우), 재무관리(홍광헌), 파생상품론(원재환), 프로세스 수익경영(이준겸), 회계학원론(김민)"
14:00~17:00,,,,,회계/재무정보분석의이해와응용(이승영)
15:00~16:15,"국제경영론(정라미), 국제마케팅론(김도의), 무역경영관리(손명옥), 세무회계(천준범), 웹데이터수집과 텍스트분석(캡스톤디자인)(김명석)","경영전략(김양민), 경영학 연구에서의 경제적 모델링(김길선), 디지털 플랫폼 비즈니스(오혜림), 빅테크 가상 시장과 하이테크 마케팅(정재학), 외환론(이강표), 유통전략론(임채운), 재무관리(이영주), 중급회계2(박두리), 창업의이해(캡스톤디자인)(김유진), 회계학원론(변상혁)","관리회계(김민), 금융리스크관리(원재환), 마케팅관리(nan), 운영관리(하병천), 투자론(홍광헌)","경영전략(김양민), 경영학 연구에서의 경제적 모델링(김길선), 디지털 플랫폼 비즈니스(오혜림), 빅테크 가상 시장과 하이테크 마케팅(정재학), 외환론(이강표), 유통전략론(임채운), 재무관리(이영주), 중급회계2(박두리), 창업의이해(캡스톤디자인)(김유진), 회계학원론(변상혁)","관리회계(김민), 금융리스크관리(원재환), 마케팅관리(nan), 시뮬레이션(정철우), 운영관리(하병천), 투자론(홍광헌)"
16:30~17:20,IT시대의 사회인준비특강(남궁훈),,,,
16:30~17:45,"경영통계학(김명석), 관리회계(양준선), 국제경영론(nan), 마케팅원론(김주영), 외감법과 공인회계사법(nan), 파생상품론(안성필)","경영 데이터사이언스(조성빈), 경영과정보기술(김아현), 금융위기와 금융산업 혁신(이상호), 마케팅원론(성연진), 중국시장및경영환경의이해(이강표), 포트폴리오관리(이영주)","경영통계학(김명석), 관리회계(양준선), 국제경영론(nan), 마케팅원론(김주영), 외감법과 공인회계사법(nan), 파생상품론(안성필)","경영 데이터사이언스(조성빈), 경영과정보기술(김아현), 금융위기와 금융산업 혁신(이상호), 마케팅원론(성연진), 중국시장및경영환경의이해(이강표), 포트폴리오관리(이영주)",


#### 혜주

In [49]:
hyeju_taken_courses = [
    "회계학원론",
    "마케팅원론", "조직행동이론", "경영통계학"
]

In [50]:
timetable_kor_hyeju = make_timetable(df_all, df_mgt_fall, hyeju_taken_courses)

timetable_kor_hyeju

요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,기술경영과기술사업화(백서현),비즈니스프로세스관리(이준겸),기술경영과기술사업화(백서현),비즈니스프로세스관리(이준겸)
10:30~11:45,"경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 모바일통신경영(이상근), 인공지능과경영모델(김진화), 인공지능과마케팅(김주영), 재무관리(김도성), 중급회계1(송민섭)","공급체인관리와e-business(김민균), 국제경영론(김창수), 기업과국제환경(김유진), 리더쉽이론(이인석), 원가회계(박예연), 인공지능 자료분석(이윤동), 투자론(박영석)","경영과학(민재형), 경영정보시스템(김용진), 기업윤리(장영균), 모바일통신경영(이상근), 인공지능과경영모델(김진화), 인공지능과마케팅(김주영), 재무관리(김도성), 중급회계1(송민섭)","공급체인관리와e-business(김민균), 국제경영론(김창수), 기업과국제환경(김유진), 리더쉽이론(이인석), 원가회계(박예연), 인공지능 자료분석(이윤동), 투자론(박영석)","경영전략(김윤진), 기업과국제환경(김장순)"
12:00~13:15,"국제경영론(정선욱), 노사관계론(양동훈), 디지털변혁과비즈니스모델혁신(김용진), 인적자원관리(조봉순), 중급회계2(송민섭), 파생상품론(이정진)","경영전략(박종훈), 경영정보시스템(오혜림), 관리회계(변상혁), 국제경영론(강리브가), 소비자행동론(성연진), 운영관리(김길선), 중급회계1(강평경), 통계자료분석(이군희)","국제경영론(정선욱), 노사관계론(양동훈), 디지털변혁과비즈니스모델혁신(김용진), 인적자원관리(조봉순), 중급회계2(송민섭), 파생상품론(이정진)","경영전략(박종훈), 경영정보시스템(오혜림), 관리회계(변상혁), 국제경영론(강리브가), 소비자행동론(성연진), 운영관리(김길선), 중급회계1(강평경), 통계자료분석(이군희)","경영전략(김윤진), 기업과국제환경(김장순)"
13:30~14:45,"국제경영론(정라미), 국제마케팅론(김도의), 무역경영관리(손명옥), 세무회계(천준범), 웹데이터수집과 텍스트분석(캡스톤디자인)(김명석)","금융기관론(이상호), 로지스틱스와생산전략(하병천), 운영관리(강세원), 원가회계(박예연), 의사결정론(조성빈), 이해관계자자본주의와ESG(박영석)","경영과학(김범수), 서비스마케팅(이민경), 재무관리(홍광헌), 파생상품론(원재환), 프로세스 수익경영(이준겸)","금융기관론(이상호), 로지스틱스와생산전략(하병천), 운영관리(강세원), 원가회계(박예연), 의사결정론(조성빈), 이해관계자자본주의와ESG(박영석)","경영과학(김범수), 서비스마케팅(이민경), 시뮬레이션(정철우), 재무관리(홍광헌), 파생상품론(원재환), 프로세스 수익경영(이준겸)"
14:00~17:00,,,,,회계/재무정보분석의이해와응용(이승영)
15:00~16:15,"국제경영론(정라미), 국제마케팅론(김도의), 무역경영관리(손명옥), 세무회계(천준범), 웹데이터수집과 텍스트분석(캡스톤디자인)(김명석)","경영전략(김양민), 경영학 연구에서의 경제적 모델링(김길선), 디지털 플랫폼 비즈니스(오혜림), 빅테크 가상 시장과 하이테크 마케팅(정재학), 외환론(이강표), 유통전략론(임채운), 재무관리(이영주), 중급회계2(박두리), 창업의이해(캡스톤디자인)(김유진)","관리회계(김민), 금융리스크관리(원재환), 마케팅관리(nan), 운영관리(하병천), 투자론(홍광헌)","경영전략(김양민), 경영학 연구에서의 경제적 모델링(김길선), 디지털 플랫폼 비즈니스(오혜림), 빅테크 가상 시장과 하이테크 마케팅(정재학), 외환론(이강표), 유통전략론(임채운), 재무관리(이영주), 중급회계2(박두리), 창업의이해(캡스톤디자인)(김유진)","관리회계(김민), 금융리스크관리(원재환), 마케팅관리(nan), 시뮬레이션(정철우), 운영관리(하병천), 투자론(홍광헌)"
16:30~17:20,IT시대의 사회인준비특강(남궁훈),,,,
16:30~17:45,"관리회계(양준선), 국제경영론(nan), 외감법과 공인회계사법(nan), 파생상품론(안성필)","경영 데이터사이언스(조성빈), 경영과정보기술(김아현), 금융위기와 금융산업 혁신(이상호), 중국시장및경영환경의이해(이강표), 포트폴리오관리(이영주)","관리회계(양준선), 국제경영론(nan), 외감법과 공인회계사법(nan), 파생상품론(안성필)","경영 데이터사이언스(조성빈), 경영과정보기술(김아현), 금융위기와 금융산업 혁신(이상호), 중국시장및경영환경의이해(이강표), 포트폴리오관리(이영주)",


### 한나

In [59]:
hanna_taken_courses = [
    "문예창작론",
    "국제인권과법",
    "고전문학자료연구",
    "디지털교육",
    "세법",
    "인문세미나",
    "교직실무",
    "형법",
    "경제학원론I",
    "프랑스언어와문화II",
    "교육행정및교육경영",
    "생명과환경",
    "문학과융합적상상력",
    "시쓰기",
    "교육방법및교육공학",
    "국어형태론",
    "문학이란무엇인가",
    "문학과문화",
    "디지털인문학강독",
    "스포츠소비자행동론",
    "국어의미론",
    "인문사회글쓰기",
    "기초인공지능프로그래밍",
    "고전소설텍스트읽기",
    "프랑스언어와문화I",
    "국어방언론",
    "현대소설론",
    "테니스",
    "현대세계와윤리문제",
    "국문학개설",
    "대학수학",
    "국어학입문",
    "파워요가",
    "그리스도교윤리",
    "국문말씨"
]


In [66]:
import re

# 1. 과목명 전처리 (괄호 제거)
def clean_course_name(name):
    return re.sub(r"\(.*?\)", "", str(name)).strip()

df_all['과목명_clean'] = df_all['과목명'].apply(clean_course_name)
name_to_code = dict(zip(df_all['과목명_clean'], df_all['과목번호']))

# 2. 전공별 과목번호 set 만들기
kor_codes_set = set(df_all.loc[df_all['과목번호'].str.startswith("KOR"), '과목번호'])
edu_codes_set = set(edu_codes)
pub_codes_set = set(pub_codes)

df_kor_selected = df_all[df_all['과목번호'].str.startswith("KOR")]
df_edu_selected = df_all[df_all['과목번호'].isin(edu_codes)]
df_pub_selected = df_all[df_all['과목번호'].isin(pub_codes)]

# 세 전공 과목 합치기
df_combined_hanna = pd.concat(
    [df_kor_selected, df_edu_selected, df_pub_selected],
    ignore_index=True
)

# Hanna 시간표 생성
timetable_hanna = make_timetable(
    df_all,
    df_combined_hanna,
    hanna_taken_courses
)
timetable_hanna

# 3. 색칠 함수
def colorize_cell(cell):
    if pd.isna(cell) or cell.strip() == "":
        return cell

    courses = str(cell).split(', ')
    colored_courses = []

    for course_name in courses:
        course_name_clean = clean_course_name(course_name)
        code = name_to_code.get(course_name_clean, None)

        in_kor = code in kor_codes_set if code else False
        in_edu = code in edu_codes_set if code else False
        in_pub = code in pub_codes_set if code else False

        # 두 개 이상 전공에 해당하면 주황색
        if sum([in_kor, in_edu, in_pub]) > 1:
            colored_courses.append(f'<span style="color:orange">{course_name}</span>')
        elif in_kor:
            colored_courses.append(f'<span style="color:red">{course_name}</span>')
        elif in_edu:
            colored_courses.append(f'<span style="color:blue">{course_name}</span>')
        elif in_pub:
            colored_courses.append(f'<span style="color:green">{course_name}</span>')
        else:
            colored_courses.append(course_name)

    return "<br>".join(colored_courses)

# 4. 적용
timetable_colored = timetable_hanna.map(colorize_cell)

from IPython.display import HTML
HTML(timetable_colored.to_html(escape=False))



요일,월,화,수,목,금
시간,,,,,
09:00~10:15,,국어교과교재연구및지도법(강효경),,국어교과교재연구및지도법(강효경),
10:30~11:20,,교육과정(양미경),,교육과정(양미경),
10:30~11:45,경제학원론II(전현배)교육학개론(신효정)국제금융론(곽준희)재무관리(김도성)행정법(임성훈)회계학원론(양준선),경영통계학(이군희)경제학원론II(곽노선)논리학개론(오은영)원가회계(박예연)투자론(박영석)한문I(백진우),경제학원론II(전현배)교육학개론(신효정)국제금융론(곽준희)재무관리(김도성)행정법(임성훈)회계학원론(양준선),경영통계학(이군희)경제학원론II(곽노선)논리학개론(오은영)원가회계(박예연)투자론(박영석)한문I(백진우),경제학원론II(김형욱)문학과문화콘텐츠(김예람)정치학개론(김태심)형사소송법(김연진)
10:30~12:20,,,생활지도및상담(계은경),,
12:00~13:15,국어의문장구조(이정훈)빅데이터의이해와교육적활용(캡스톤디자인)(신효정)정치학개론(이현우),거시경제학I(이윤수)관리회계(변상혁)국어교과논리및논술(강효경)현대소설텍스트읽기II(김경수)회계학원론(박두리),국어의문장구조(이정훈)빅데이터의이해와교육적활용(캡스톤디자인)(신효정)정치학개론(이현우),거시경제학I(이윤수)관리회계(변상혁)국어교과논리및논술(강효경)현대소설텍스트읽기II(김경수)회계학원론(박두리),경제학원론II(김형욱)문학과문화콘텐츠(김예람)정치학개론(김태심)형사소송법(김연진)
12:00~14:45,,,경제법(이나정),,
13:30~14:45,"경제학원론II(김숙영)민법 Ⅱ (채권총칙,채권각칙,가족법)(이승기)일반심리학(홍승범)",거시경제학I(허준영)경영통계학(이윤동)삶과교육(양미경)원가회계(박예연)한문과한문학(백진우),사회학개론(김영수)소설과영화(김예람)재무관리(홍광헌)회계학원론(김민),거시경제학I(허준영)경영통계학(이윤동)삶과교육(양미경)원가회계(박예연)한문과한문학(백진우),국문학과전통문화(최지혜)사회학개론(김영수)소설과영화(김예람)재무관리(홍광헌)지식재산권법(김미화)회계학원론(김민)
13:30~15:20,교육사회(김영미),,,,
15:00~16:15,"경제학원론II(김숙영)민법 Ⅱ (채권총칙,채권각칙,가족법)(이승기)일반심리학(홍승범)",경제학원론II(허준영)국어학연습(김한별)노동법(심재진)사회학개론(민병교)이야기와이야기창작(김경수)재무관리(이영주)회계학원론(변상혁),관리회계(김민)국어담화화용론(명정희)투자론(홍광헌),경제학원론II(허준영)국어학연습(김한별)노동법(심재진)사회학개론(민병교)이야기와이야기창작(김경수)재무관리(이영주)회계학원론(변상혁),관리회계(김민)국문학과전통문화(최지혜)국어담화화용론(명정희)지식재산권법(김미화)투자론(홍광헌)
